In [1]:
import pandas as pd
 
IN_PATH = "WA_Unemployment_2015_2024.csv"
REF_PATH = "housing_final.csv"
OUT_PATH = "cleaned_unemployment_rate.csv"
 
START_YEAR, END_YEAR = 2015, 202

In [2]:
# Load data set
df = pd.read_csv(IN_PATH, dtype={"CountyFIPS": str})

# County -> WSDOT region lookup taken from housing_final (one row per county)
ref = (pd.read_csv(REF_PATH, dtype={"county_fips": str}, encoding="utf-8-sig")
         [["county_fips", "county", "wsdot_region"]]
         .drop_duplicates("county_fips"))
print("Reference counties:", len(ref), "| WSDOT regions:", ref["wsdot_region"].nunique())

Reference counties: 39 | WSDOT regions: 6


In [3]:
# Clean data set
df = df.rename(columns={"CountyFIPS": "county_fips", "County": "county_raw", "Year": "year", "UnemploymentRate": "unemployment_rate"})
df["county_fips"] = df["county_fips"].str.strip().str.zfill(5)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["unemployment_rate"] = pd.to_numeric(df["unemployment_rate"], errors="coerce")

# "Adams County, Washington" -> "Adams County" (same format as housing_final)
df["county_raw"] = df["county_raw"].str.replace(", Washington", "", regex=False).str.strip()

n_raw = len(df)
df = (df.dropna(subset=["county_fips", "year", "unemployment_rate"])
        .drop_duplicates(["county_fips", "year"])
        .query("@START_YEAR <= year <= @END_YEAR", engine="python"))
print(f"Rows: {n_raw} raw -> {len(df)} after cleaning")

Rows: 390 raw -> 0 after cleaning


In [4]:
# Assign WSDOT region (merge on FIPS)
df = df.merge(ref, on="county_fips", how="left", validate="many_to_one")

# Every county must match the reference, and the names must agree
assert df["wsdot_region"].notna().all(), "Some counties were not found in housing_final"
mismatch = df[df["county_raw"] != df["county"]]
assert mismatch.empty, "County names differ from housing_final: " + str(mismatch["county_fips"].unique())

df = (df[["county_fips", "county", "wsdot_region", "year", "unemployment_rate"]]
        .sort_values(["county_fips", "year"]).reset_index(drop=True))
df["year"] = df["year"].astype(int)

In [5]:
# Validate
assert df["county_fips"].nunique() == 39, "Expected 39 counties"
assert df.groupby("county_fips")["year"].nunique().eq(10).all(), "Every county needs all 10 years"
assert len(df) == 390
assert df["unemployment_rate"].between(0, 40).all()

# Cross-check against the unemployment column already in housing_final
hf = pd.read_csv(REF_PATH, dtype={"county_fips": str}, encoding="utf-8-sig")[["county_fips", "year", "unemployment_rate"]]
chk = df.merge(hf, on=["county_fips", "year"], suffixes=("", "_housing"), validate="one_to_one")
print("Rows matched to housing_final:", len(chk), "| rates that differ:", (chk["unemployment_rate"] != chk["unemployment_rate_housing"]).sum())

print("\nCounties per WSDOT region:")
print(df.drop_duplicates("county_fips")["wsdot_region"].value_counts().to_string())

AssertionError: Expected 39 counties

In [ ]:
# Save
df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUT_PATH}")
df.head(12)

In [ ]:
df.groupby("wsdot_region")["unemployment_rate"].agg(["count", "mean", "min", "max"]).round(2)